# Treatment-effect results: analysis and visualisation

**Analysis only:** this notebook loads saved datasets and predictions, aggregates results, and creates plots. Run model fitting separately with `python runnables/run_treatment_effect_experiments.py`.

This analysis notebook examines how estimated treatment effects vary with a patient's pre-treatment history. It follows two stages: **first stage — nuisance models**, which estimate treatment and outcome probabilities, and **second stage — CATE learner**, which regresses constructed treatment-effect targets on patient features. Run experiments with `python runnables/run_treatment_effect_experiments.py`; this notebook loads their saved outputs and compares pretrained and scratch transformers with tabular baselines, then examines internal calibration.

### Target and notation

Let $X$ denote the pre-index patient history and demographic features, $A\in\{0,1\}$ the treatment indicator (1: treatment; 0: comparator), and $Y$ the binary outcome within the configured follow-up window. With potential outcomes $Y(1)$ and $Y(0)$, the conditional average treatment effect is

$$\tau(x)=\mathbb{E}[Y(1)-Y(0)\mid X=x].$$

For a binary outcome this is a conditional risk difference. A negative value means lower outcome risk under treatment than under the comparator. A CATE describes an average effect conditional on features; it does not reveal an individual's unobserved counterfactual outcome.


### How to run the tutorial

The notebook loads saved prediction files directly. Run missing stages with `runnables/run_treatment_effect_experiments.py` before analysis. The runnable reuses cached datasets, splits, and model predictions; use a new output directory when changing study or model settings. Missing saved outputs raise a loading error.

The sequence is dataset preparation → first-stage cross-fitting → pseudo-outcomes and nuisance diagnostics → second-stage CATE learners → internal effect-score calibration.

In [ ]:

from pathlib import Path
import importlib.metadata
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Find the repository from a root or nested notebook working directory.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'DR_Learner').is_dir() and (path / 'src').is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from the repository or one of its subdirectories.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from runnables.treatment_effect_config import build_config

# Report the execution environment without changing it.
print(f'Python: {sys.version.split()[0]}')
for package in ('torch', 'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', 'pyarrow', 'xgboost'):
    try:
        print(f'{package}: {importlib.metadata.version(package)}')
    except importlib.metadata.PackageNotFoundError:
        print(f'{package}: not installed')


## 1. Select saved results

Run the experiment from the repository root before using this notebook:

```sh
python runnables/run_treatment_effect_experiments.py --experiment glp1_vs_dpp4
```

Set `EXPERIMENT`, `SEEDS`, and optionally `OUTPUT_DIR` below to match that run. Model and study defaults live in `runnables/treatment_effect_config.py`. This notebook never fits models or builds datasets; missing outputs produce a loading error.


In [ ]:
EXPERIMENT = 'glp1_vs_dpp4'
SEEDS = (17, 24, 51, 103, 204)
OUTPUT_DIR = None  # Set to the runner's --output-dir if you used a custom directory.
config = build_config(experiment=EXPERIMENT, seeds=SEEDS, output_dir=OUTPUT_DIR)
globals().update(vars(config))
print(f'Loading results: {RESULT_DIR}')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)


### Aggregation helpers

Summaries report the across-seed mean, standard deviation, and standard error.


In [ ]:
def summarize_metric_series(values: pd.Series, z_value: float=1.96) -> pd.Series:
    numeric_values = pd.to_numeric(values, errors='coerce').dropna().to_numpy(dtype=float)
    n = int(len(numeric_values))
    if n == 0:
        return pd.Series({'mean': np.nan, 'std': np.nan, 'se': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n': 0})
    mean = float(numeric_values.mean())
    std = float(numeric_values.std(ddof=1)) if n > 1 else 0.0
    se = float(std / np.sqrt(n)) if n > 1 else 0.0
    half_width = z_value * se
    return pd.Series({'mean': mean, 'std': std, 'se': se, 'ci_low': mean - half_width, 'ci_high': mean + half_width, 'n': n})

def summarize_seed_distribution(df: pd.DataFrame, group_cols: list[str], value_cols: list[str], z_value: float=1.96) -> pd.DataFrame:
    summary_frames = []
    grouped = df.groupby(group_cols, dropna=False, observed=True)
    for value_col in value_cols:
        summary_df = grouped[value_col].apply(lambda values: summarize_metric_series(values, z_value=z_value))
        summary_df = summary_df.unstack(level=-1)
        summary_df.columns = [f'{value_col}_{suffix}' for suffix in summary_df.columns]
        summary_frames.append(summary_df)
    if not summary_frames:
        return grouped.size().reset_index(name='n').drop(columns='n')
    return pd.concat(summary_frames, axis=1).reset_index()


## 2. Load the saved analysis population

The next cell loads the indexed dataset and split saved by the experiment runner. Each row represents a patient indexed at the earliest qualifying treatment/comparator exposure. The builder constructs the pre-index event history, assigns treatment, and records the outcome within the configured post-index window. Review these definitions before starting a new study: they determine the population and estimand.

The development/test split is stratified by treatment and outcome. All following fits and diagnostics use the **development population**.

In [ ]:
samples_df = pd.read_parquet(SAMPLES_CACHE_PATH).reset_index(drop=True)
split_payload = json.loads(TRAIN_TEST_SPLIT_PATH.read_text())
development_samples_df = samples_df.iloc[split_payload['train']].reset_index(drop=True).copy()
heldout_test_samples_df = samples_df.iloc[split_payload['test']].reset_index(drop=True).copy()
display(pd.DataFrame([
    {'split': name, 'n': len(frame), 'treated_fraction': frame['treatment'].mean(), 'outcome_fraction': frame['outcome'].mean()}
    for name, frame in [('development', development_samples_df), ('reserved_test', heldout_test_samples_df)]
]))


## 3. Load and summarise saved nuisance predictions

The first stage estimates the propensity score and the treatment-specific outcome regressions:

$$e(x)=P(A=1\mid X=x),\qquad \mu_a(x)=\mathbb{E}[Y\mid A=a,X=x],\quad a\in\{0,1\}.$$

The pretrained transformer learns propensity and factual-outcome predictions from patient histories. Evaluating its outcome model under each treatment value yields $\hat\mu_0$ and $\hat\mu_1$. The scratch transformer provides a comparison without encoder pretraining. The training settings specify the relative contribution of the propensity and outcome objectives.

**Cross-fitting:** partition development rows into $K$ folds. For each fold, fit nuisance models using the other folds, with an internal validation split, and predict the excluded rows. Combining those predictions gives one out-of-fold nuisance estimate per development patient. Thus, a row's first-stage targets are constructed from models that did not fit that row.

The next cells load each seed's saved pretrained and scratch predictions and metrics.

In [ ]:
scratch_transformer_seed_prediction_map = {}
scratch_transformer_seed_fold_metrics = []
scratch_transformer_seed_summary_rows = []
for seed in SEEDS:
    seed_dir = SCRATCH_TRANSFORMER_NUISANCE_ROOT / f'seed_{seed}'
    predictions_df = pd.read_parquet(seed_dir / 'patient_nuisance_predictions.parquet')
    fold_metrics_df = pd.read_parquet(seed_dir / 'fold_metrics.parquet')
    metrics_payload = json.loads((seed_dir / 'metrics.json').read_text())
    scratch_transformer_seed_prediction_map[seed] = predictions_df
    fold_metrics_df['seed'] = seed
    scratch_transformer_seed_fold_metrics.append(fold_metrics_df)
    scratch_transformer_seed_summary_rows.append({'seed': seed, **metrics_payload['summary']})
scratch_transformer_seed_fold_metrics_df = pd.concat(scratch_transformer_seed_fold_metrics, ignore_index=True)
scratch_transformer_seed_summary_df = pd.DataFrame(scratch_transformer_seed_summary_rows).sort_values('seed').reset_index(drop=True)
print(f'Scratch transformer nuisance baselines saved to: {SCRATCH_TRANSFORMER_NUISANCE_ROOT}')
display(scratch_transformer_seed_summary_df)
display(scratch_transformer_seed_fold_metrics_df.groupby('seed')[['holdout/loss', 'holdout/propensity_auc', 'holdout/outcome_auc']].mean())


In [ ]:
nuisance_seed_predictions = []
nuisance_seed_fold_metrics = []
nuisance_seed_summary_rows = []

for seed in SEEDS:
    seed_dir = NUISANCE_ROOT / f'seed_{seed}'
    predictions_df = pd.read_parquet(seed_dir / 'patient_nuisance_predictions.parquet')
    fold_metrics_df = pd.read_parquet(seed_dir / 'fold_metrics.parquet')
    metrics_payload = json.loads((seed_dir / 'metrics.json').read_text())

    predictions_df['seed'] = seed
    fold_metrics_df['seed'] = seed
    nuisance_seed_predictions.append(predictions_df)
    nuisance_seed_fold_metrics.append(fold_metrics_df)

    nuisance_seed_summary_rows.append({
        'seed': seed,
        **metrics_payload['summary'],
    })

nuisance_seed_predictions_df = pd.concat(nuisance_seed_predictions, ignore_index=True)
nuisance_seed_fold_metrics_df = pd.concat(nuisance_seed_fold_metrics, ignore_index=True)
nuisance_seed_summary_df = pd.DataFrame(nuisance_seed_summary_rows).sort_values('seed').reset_index(drop=True)

prediction_mean_columns = [
    'propensity',
    'propensity_clipped',
    'outcome_probability_observed_treatment',
    'outcome_probability_treatment_0',
    'outcome_probability_treatment_1',
    'individual_treatment_effect',
    'ipw_pseudo_outcome',
    'aipw_pseudo_outcome',
    'dr_pseudo_outcome',
]
static_columns = [
    column
    for column in nuisance_seed_predictions_df.columns
    if column not in {'seed', *prediction_mean_columns}
]

propensity_static_df = (
    nuisance_seed_predictions_df
    .sort_values(['row_index', 'seed'])
    .groupby('row_index', as_index=False)[static_columns]
    .first()
)
propensity_prediction_mean_df = (
    nuisance_seed_predictions_df
    .groupby('row_index', as_index=False)[prediction_mean_columns]
    .mean()
)
propensity_prediction_std_df = (
    nuisance_seed_predictions_df
    .groupby('row_index', as_index=False)[prediction_mean_columns]
    .std(ddof=1)
    .rename(columns={column: f'{column}_seed_std' for column in prediction_mean_columns})
)

propensity_df = (
    propensity_static_df
    .merge(propensity_prediction_mean_df, on='row_index', how='left', validate='one_to_one')
    .merge(propensity_prediction_std_df, on='row_index', how='left', validate='one_to_one')
)
propensity_df['nuisance_individual_treatment_effect'] = propensity_df['individual_treatment_effect']
propensity_df['analysis_split'] = 'development_train'

display(nuisance_seed_summary_df)
display(
    nuisance_seed_fold_metrics_df.groupby('seed')[['holdout/loss', 'holdout/propensity_auc', 'holdout/outcome_auc']].mean()
)
display(propensity_df.head(3))


### Analyse saved IPW and AIPW pseudo-outcomes

The first-stage predictions are converted into a regression target for each patient. Write $\tilde e_i$ for the estimated propensity after clipping. 

The AIPW target adds outcome predictions and weighted residual corrections:

$$\phi_i^{\mathrm{AIPW}}=\hat\mu_1(X_i)-\hat\mu_0(X_i)
+\frac{A_i\{Y_i-\hat\mu_1(X_i)\}}{\tilde e_i}
-\frac{(1-A_i)\{Y_i-\hat\mu_0(X_i)\}}{1-\tilde e_i}.$$

Under identification assumptions, the AIPW target has conditional mean $\tau(x)$ if either the propensity or both outcome regressions are correct. 


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pseudo_outcome_columns = ['aipw_pseudo_outcome'] #'ipw_pseudo_outcome',
pseudo_outcome_summary_df = (
    nuisance_seed_predictions_df
    .groupby('seed')[pseudo_outcome_columns]
    .agg(['mean', 'std', 'median'])
)
display(pseudo_outcome_summary_df)

reference_pseudo_outcomes_df = nuisance_seed_predictions_df.loc[
    nuisance_seed_predictions_df['seed'].eq(REFERENCE_SEED),
    ['seed', *pseudo_outcome_columns],
].copy()
if reference_pseudo_outcomes_df.empty:
    raise ValueError(f'No nuisance predictions found for REFERENCE_SEED={REFERENCE_SEED}.')

reference_pseudo_outcome_summary_df = (
    reference_pseudo_outcomes_df[pseudo_outcome_columns]
    .agg(['mean', 'std', 'median'])
    .T
)
display(reference_pseudo_outcome_summary_df)


def histogram_bins(values: np.ndarray, min_bins: int = 25, max_bins: int = 70) -> int:
    edges = np.histogram_bin_edges(values, bins='fd')
    if len(edges) <= 2:
        return min_bins
    return int(np.clip(len(edges) - 1, min_bins, max_bins))


def clip_distribution_tails(values: np.ndarray, lower_q: float = 0.01, upper_q: float = 0.99) -> tuple[np.ndarray, float, float]:
    lower_bound, upper_bound = np.quantile(values, [lower_q, upper_q])
    clipped_values = values[(values >= lower_bound) & (values <= upper_bound)]
    return clipped_values, float(lower_bound), float(upper_bound)


plot_labels = {
    'ipw_pseudo_outcome': 'IPW pseudo-outcome',
    'aipw_pseudo_outcome': 'AIPW pseudo-outcome',
}
plot_colors = {
    'ipw_pseudo_outcome': '#2563eb',
    'aipw_pseudo_outcome': '#4C9F9C',
}

cap_percentile = 99.5

with plt.style.context('seaborn-v0_8-whitegrid'):
    fig, ax = plt.subplots(
        figsize=(5, 4),
        dpi=140,
        constrained_layout=True
    )

    values = reference_pseudo_outcomes_df[pseudo_outcome_columns[0]].dropna().to_numpy(dtype=float)

    clipped_values, lower_bound, upper_bound = clip_distribution_tails(values)

    # Statistics calculated on ALL clipped values
    mean_value = clipped_values.mean()
    median_value = np.median(clipped_values)

    ax.hist(
        clipped_values,
        bins=(histogram_bins(clipped_values)-10),
        color=plot_colors[pseudo_outcome_columns[0]],
        alpha=0.86,
        edgecolor='white',
        linewidth=0.7,
    )

    ax.axvline(
        mean_value,
        color='#202124',
        linestyle='--',
        linewidth=1.25,
        label=f'Mean: {mean_value:.4f}',
    )

    # Only restrict the DISPLAYED x-axis
    if cap_percentile is not None:
        display_max = np.percentile(clipped_values, cap_percentile)
        ax.set_xlim(right=display_max)

    ax.set_xlabel('Pseudo-outcome value')
    ax.set_ylabel('Patients')

    ax.grid(axis='y', alpha=0.28)
    ax.grid(axis='x', visible=False)

    # Clean spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.tick_params(axis='y', length=0)

    ax.legend(frameon=False, fontsize=9)
    output_path = FIGURE_DIR / 'pseudo_outcomes_distribution.svg'

    fig.savefig(output_path, format='svg', bbox_inches='tight')
    plt.show()


### Load and compare saved tabular baseline results

The following cells compare the transformer nuisance models with random forest, XGBoost, and an age/sex baseline. Each baseline estimates treatment probabilities and treatment-specific outcome probabilities, then constructs the same pseudo-outcomes.

The experiment runner fits the baselines; the cells below load their results. Baselines reuse the pretrained transformer's saved fold indices to compare predictions on the same excluded patients. 


In [ ]:
rf_seed_prediction_map = {}
rf_seed_fold_metrics = []
rf_seed_summary_rows = []
xgb_seed_prediction_map = {}
xgb_seed_fold_metrics = []
xgb_seed_summary_rows = []
age_sex_seed_prediction_map = {}
age_sex_seed_fold_metrics = []
age_sex_seed_summary_rows = []
for seed in SEEDS:
    rf_seed_output_dir = RF_OUTPUT_DIR / f'seed_{seed}'
    rf_prediction_path = rf_seed_output_dir / 'patient_nuisance_predictions.parquet'
    rf_fold_metrics_path = rf_seed_output_dir / 'fold_metrics.parquet'
    print(f'Skipping RF nuisance baseline seed {seed}; found {rf_prediction_path}')
    rf_prediction_df = pd.read_parquet(rf_prediction_path)
    rf_fold_metrics_df = pd.read_parquet(rf_fold_metrics_path)
    rf_seed_prediction_map[seed] = rf_prediction_df
    rf_seed_fold_metrics.append(rf_fold_metrics_df.assign(seed=seed))
    rf_seed_summary_rows.append({'seed': seed, 'mean_propensity': float(rf_prediction_df['propensity'].mean()), 'mean_individual_treatment_effect': float(rf_prediction_df['individual_treatment_effect'].mean()), 'ipw_ate': float(rf_prediction_df['ipw_pseudo_outcome'].mean()), 'aipw_ate': float(rf_prediction_df['aipw_pseudo_outcome'].mean()), 'mean_abs_aipw_pseudo_outcome': float(rf_prediction_df['aipw_pseudo_outcome'].abs().mean())})
for seed in SEEDS:
    xgb_seed_output_dir = XGB_OUTPUT_DIR / f'seed_{seed}'
    xgb_prediction_path = xgb_seed_output_dir / 'patient_nuisance_predictions.parquet'
    xgb_fold_metrics_path = xgb_seed_output_dir / 'fold_metrics.parquet'
    print(f'Skipping XGBoost nuisance baseline seed {seed}; found {xgb_prediction_path}')
    xgb_prediction_df = pd.read_parquet(xgb_prediction_path)
    xgb_fold_metrics_df = pd.read_parquet(xgb_fold_metrics_path)
    xgb_seed_prediction_map[seed] = xgb_prediction_df
    xgb_seed_fold_metrics.append(xgb_fold_metrics_df.assign(seed=seed))
    xgb_seed_summary_rows.append({'seed': seed, 'mean_propensity': float(xgb_prediction_df['propensity'].mean()), 'mean_individual_treatment_effect': float(xgb_prediction_df['individual_treatment_effect'].mean()), 'ipw_ate': float(xgb_prediction_df['ipw_pseudo_outcome'].mean()), 'aipw_ate': float(xgb_prediction_df['aipw_pseudo_outcome'].mean()), 'mean_abs_aipw_pseudo_outcome': float(xgb_prediction_df['aipw_pseudo_outcome'].abs().mean())})
for seed in SEEDS:
    age_sex_seed_output_dir = AGE_SEX_OUTPUT_DIR / f'seed_{seed}'
    age_sex_prediction_path = age_sex_seed_output_dir / 'patient_nuisance_predictions.parquet'
    age_sex_fold_metrics_path = age_sex_seed_output_dir / 'fold_metrics.parquet'
    print(f'Skipping age/sex nuisance baseline seed {seed}; found {age_sex_prediction_path}')
    age_sex_prediction_df = pd.read_parquet(age_sex_prediction_path)
    age_sex_fold_metrics_df = pd.read_parquet(age_sex_fold_metrics_path)
    age_sex_seed_prediction_map[seed] = age_sex_prediction_df
    age_sex_seed_fold_metrics.append(age_sex_fold_metrics_df.assign(seed=seed))
    age_sex_seed_summary_rows.append({'seed': seed, 'mean_propensity': float(age_sex_prediction_df['propensity'].mean()), 'mean_individual_treatment_effect': float(age_sex_prediction_df['individual_treatment_effect'].mean()), 'ipw_ate': float(age_sex_prediction_df['ipw_pseudo_outcome'].mean()), 'aipw_ate': float(age_sex_prediction_df['aipw_pseudo_outcome'].mean()), 'mean_abs_aipw_pseudo_outcome': float(age_sex_prediction_df['aipw_pseudo_outcome'].abs().mean())})
rf_seed_fold_metrics_df = pd.concat(rf_seed_fold_metrics, ignore_index=True)
rf_seed_summary_df = pd.DataFrame(rf_seed_summary_rows).sort_values('seed').reset_index(drop=True)
xgb_seed_fold_metrics_df = pd.concat(xgb_seed_fold_metrics, ignore_index=True)
xgb_seed_summary_df = pd.DataFrame(xgb_seed_summary_rows).sort_values('seed').reset_index(drop=True)
age_sex_seed_fold_metrics_df = pd.concat(age_sex_seed_fold_metrics, ignore_index=True)
age_sex_seed_summary_df = pd.DataFrame(age_sex_seed_summary_rows).sort_values('seed').reset_index(drop=True)

def fold_metrics_long_to_wide(fold_metrics_df: pd.DataFrame) -> pd.DataFrame:
    index_columns = ['seed', 'fold']
    wide_df = fold_metrics_df[index_columns + ['task', 'holdout/loss', 'holdout/auc']].pivot(index=index_columns, columns='task', values=['holdout/loss', 'holdout/auc']).reset_index()
    flattened_columns = []
    for column in wide_df.columns.to_flat_index():
        if isinstance(column, tuple):
            left, right = column
            if right in ('', None):
                flattened_columns.append(str(left))
            else:
                flattened_columns.append(f'{left}__{right}')
        else:
            flattened_columns.append(str(column))
    wide_df.columns = flattened_columns
    wide_df = wide_df.rename(columns={'holdout/loss__propensity': 'holdout/propensity_loss', 'holdout/loss__outcome': 'holdout/outcome_loss', 'holdout/auc__propensity': 'holdout/propensity_auc', 'holdout/auc__outcome': 'holdout/outcome_auc'})
    required_columns = ['seed', 'fold', 'holdout/propensity_loss', 'holdout/outcome_loss', 'holdout/propensity_auc', 'holdout/outcome_auc']
    missing_columns = [column for column in required_columns if column not in wide_df.columns]
    if missing_columns:
        raise KeyError(f'Baseline fold metric reshape is missing columns: {missing_columns}')
    wide_df['holdout/loss'] = wide_df[['holdout/propensity_loss', 'holdout/outcome_loss']].mean(axis=1)
    return wide_df

def build_model_fold_comparison_frame(model_name: str, fold_metrics_df: pd.DataFrame) -> pd.DataFrame:
    return fold_metrics_df.assign(model=model_name)[['model', 'seed', 'fold', 'holdout/loss', 'holdout/propensity_auc', 'holdout/outcome_auc']]
rf_fold_metrics_wide_df = fold_metrics_long_to_wide(rf_seed_fold_metrics_df)
xgb_fold_metrics_wide_df = fold_metrics_long_to_wide(xgb_seed_fold_metrics_df)
age_sex_fold_metrics_wide_df = fold_metrics_long_to_wide(age_sex_seed_fold_metrics_df)
nuisance_model_comparison_frames = [build_model_fold_comparison_frame('ITE-Transformer', nuisance_seed_fold_metrics_df), build_model_fold_comparison_frame('Age + sex', age_sex_fold_metrics_wide_df), build_model_fold_comparison_frame('Random forest', rf_fold_metrics_wide_df), build_model_fold_comparison_frame('XGBoost', xgb_fold_metrics_wide_df)]
nuisance_model_fold_comparison_df = pd.concat(nuisance_model_comparison_frames, ignore_index=True)
nuisance_model_holdout_summary_df = nuisance_model_fold_comparison_df.groupby('model')[['holdout/loss', 'holdout/propensity_auc', 'holdout/outcome_auc']].apply(lambda frame: pd.concat({column: summarize_metric_series(frame[column]) for column in frame.columns}, axis=0)).unstack(level=-1)
nuisance_seed_level_summary_df = pd.concat([nuisance_seed_summary_df.assign(model='ITE-Transformer'), age_sex_seed_summary_df.assign(model='Age + sex'), rf_seed_summary_df.assign(model='Random forest'), xgb_seed_summary_df.assign(model='XGBoost')], ignore_index=True)
nuisance_effect_summary_df = nuisance_seed_level_summary_df.groupby('model')[['mean_propensity', 'mean_individual_treatment_effect', 'ipw_ate', 'aipw_ate', 'mean_abs_aipw_pseudo_outcome']].apply(lambda frame: pd.concat({column: summarize_metric_series(frame[column]) for column in frame.columns}, axis=0)).unstack(level=-1)
display(nuisance_model_fold_comparison_df)
display(nuisance_model_holdout_summary_df)
display(nuisance_effect_summary_df)
print(f'Age/sex nuisance baselines saved to: {AGE_SEX_OUTPUT_DIR}')
print(f'Random forest nuisance baselines saved to: {RF_OUTPUT_DIR}')
print(f'XGBoost nuisance baselines saved to: {XGB_OUTPUT_DIR}')


## 4. Assess first-stage predictive performance

The panels compare out-of-fold propensity and factual-outcome predictions using AUROC and AUPRC. AUROC describes discrimination across thresholds; AUPRC emphasises precision and recall for the positive class and should be read alongside class prevalence. The mean-prediction baseline provides a reference.

`NP-Transformer` labels the non pretrained transformer. Any “held-out” labels in these panels refer to excluded cross-fitting folds within development. Across-seed bands summarise run variability.

These comparisons assess nuisance prediction quality which are important for the reliability  of the pseudo-outcomes.


In [ ]:
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score
from src.DR_Learner.helpers import binary_auc


MODEL_ORDER = [
    'ITE-Transformer',
    'XGBoost',
    'Random forest',
    'NP-Transformer',
    'Mean predictor',
    'Age + sex',

]
PLOT_MODEL_ORDER = [model for model in MODEL_ORDER if model != 'Mean predictor']
MODEL_COLORS = {
    'Mean predictor': '#8A8F98',
    'Age + sex': '#5B8DEF',
    'Random forest': '#3D5A80',
    'XGBoost': '#B565A7',
    'ITE-Transformer': '#4C9F9C',#4C9F9C',#'#5B8DEF',
    'NP-Transformer': '#E07A5F',#'#2A9D8F', #B565A7
}
PLOT_METRICS = [
    ('auc', 'AUROC', (0.0, 1.0)),
    ('pr_auc', 'AUPRC', (0.0, 1.0)),
]

METRIC_FILE_LABELS = {
    'auc': 'auroc',
    'pr_auc': 'auprc',
}


def safe_filename_token(value: str) -> str:
    return ''.join(
        char.lower() if char.isalnum() else '_'
        for char in str(value)
    ).strip('_')


METRIC_PLOT_DIR = FIGURE_DIR
METRIC_PLOT_DIR.mkdir(parents=True, exist_ok=True)


def classification_metrics_from_probabilities(labels, probabilities, pos_weight=None):
    labels_np = np.asarray(labels, dtype=np.float64)
    probs = np.clip(np.asarray(probabilities, dtype=np.float64), 1e-6, 1 - 1e-6)


    if pos_weight is None:
        losses = -(labels_np * np.log(probs) + (1.0 - labels_np) * np.log(1.0 - probs))
    else:
        losses = -(pos_weight * labels_np * np.log(probs) + (1.0 - labels_np) * np.log(1.0 - probs))

    predictions = (probs >= 0.5).astype(int)
    labels_int = labels_np.astype(int)
    loss = float(losses.mean())

    return {
        'loss': loss,
        'log2_loss': float(loss / np.log(2.0)),
        'accuracy': float((predictions == labels_int).mean()),
        'precision': float(precision_score(labels_int, predictions, zero_division=0)),
        'recall': float(recall_score(labels_int, predictions, zero_division=0)),
        'f1': float(f1_score(labels_int, predictions, zero_division=0)),
        'auc': binary_auc(labels_int.tolist(), probs.tolist()),
        'pr_auc': float(average_precision_score(labels_int, probs)),
    }


def constant_prediction_metrics(labels, probability, pos_weight=None):
    labels_np = np.asarray(labels, dtype=np.float64)
    probs = np.full(labels_np.shape, float(np.clip(probability, 1e-6, 1 - 1e-6)), dtype=np.float64)
    return classification_metrics_from_probabilities(
        labels=labels_np,
        probabilities=probs,
        pos_weight=pos_weight,
    )


def build_comparison_df(metric_rows):
    return pd.DataFrame([
        {'model': model_name, **metrics}
        for model_name, metrics in metric_rows
    ])


def sort_models(df):
    return df.assign(
        _model_order=df['model'].map({model: idx for idx, model in enumerate(MODEL_ORDER)})
    ).sort_values('_model_order').drop(columns='_model_order')


def barplot_with_ci(ax, df, metric_key, title, ylim=None):
    plot_df = sort_models(
        df.loc[df['model'].isin(PLOT_MODEL_ORDER)].copy()
    ).iloc[::-1].reset_index(drop=True)
    value_col = f'{metric_key}_mean' if f'{metric_key}_mean' in plot_df.columns else metric_key
    low_col = f'{metric_key}_ci_low' if f'{metric_key}_ci_low' in plot_df.columns else None
    high_col = f'{metric_key}_ci_high' if f'{metric_key}_ci_high' in plot_df.columns else None

    x = np.arange(len(plot_df)) * 1.1
    values = plot_df[value_col].to_numpy(dtype=float)
    colors = [MODEL_COLORS.get(model_name, '#8A8F98') for model_name in plot_df['model']]
    ax.bar(x, values, width=0.58, color=colors, zorder=2)


    lower_bounds = values.copy()
    upper_bounds = values.copy()
    if low_col is not None and high_col is not None:
        lower = values - plot_df[low_col].to_numpy(dtype=float)
        upper = plot_df[high_col].to_numpy(dtype=float) - values
        lower_bounds = plot_df[low_col].to_numpy(dtype=float)
        upper_bounds = plot_df[high_col].to_numpy(dtype=float)
        ax.errorbar(
            x,
            values,
            yerr=np.vstack([lower, upper]),
            fmt='none',
            ecolor='#374151',
            elinewidth=1.2,
            capsize=4,
            zorder=3,
        )

    ax.set_title('')
    ax.set_xlabel('')
    ax.set_ylabel(title)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['model'], rotation=0)#, ha='right')
    ax.tick_params(axis='x', which='both', length=0)
    ax.tick_params(axis='y', which='both', length=0)
    ax.margins(x=0.08)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('grey')
    ax.spines['bottom'].set_color('grey')

    y_lower = float(np.nanmin(lower_bounds))
    y_upper = float(np.nanmax(upper_bounds))
    y_span = max(y_upper - y_lower, 0.01)
    y_pad = y_span * 0.28

    if ylim is not None:
        hard_min, hard_max = ylim
        plot_min = max(hard_min, y_lower - y_pad)
        plot_max = min(hard_max, y_upper + y_pad)
        if plot_max <= plot_min:
            plot_min, plot_max = hard_min, hard_max
    else:
        plot_min = y_lower - y_pad
        plot_max = y_upper + y_pad

    ax.set_ylim(plot_min, plot_max)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax.grid(axis='y', color='#D1D5DB', linewidth=0.8, alpha=0.8)
    ax.set_axisbelow(True)

    label_offset = max((plot_max - plot_min) * 0.025, 0.004)
    for xpos, value, low, high in zip(x, values, lower_bounds, upper_bounds):
        label_y = high + label_offset
        va = 'bottom'
        if label_y > plot_max:
            label_y = low - label_offset
            va = 'top'
        ax.text(xpos, label_y, f'{value:.3f}', ha='center', va=va, fontsize=9, clip_on=False)


def plot_comparison_metrics(df, output_prefix):
    fig, axes = plt.subplots(len(PLOT_METRICS), 1, figsize=(7, 9), dpi=140)
    for ax, (metric_key, metric_label, limit) in zip(np.ravel(axes), PLOT_METRICS):
        barplot_with_ci(ax, df, metric_key, metric_label, ylim=limit)
    fig.tight_layout(rect=[0, 0, 1, 0.94], w_pad=3.0)
    output_path = METRIC_PLOT_DIR / f'{output_prefix}_bar.svg'
    fig.savefig(output_path, format='svg', bbox_inches='tight')
    print(f'Saved {output_path}')
    plt.show()


def observed_outcome_probabilities(prediction_df):
    return np.where(
        prediction_df['treatment'].to_numpy(dtype=int) == 1,
        prediction_df['outcome_probability_treatment_1'].to_numpy(dtype=float),
        prediction_df['outcome_probability_treatment_0'].to_numpy(dtype=float),
    )


def build_mean_predictor_oof(seed: int) -> tuple[np.ndarray, np.ndarray]:
    path = BASELINE_OUTPUT_ROOT / 'mean_predictor' / f'seed_{seed}' / 'patient_nuisance_predictions.parquet'
    predictions = pd.read_parquet(path).sort_values('row_index').reset_index(drop=True)
    expected = np.sort(development_samples_df['row_index'].to_numpy(dtype=int))
    if not np.array_equal(predictions['row_index'].to_numpy(dtype=int), expected):
        raise ValueError(f'Mean-predictor predictions for seed {seed} do not match development rows.')
    return (predictions['propensity'].to_numpy(dtype=float),
            predictions['outcome_probability_observed_treatment'].to_numpy(dtype=float))


comparison_metric_cols = ['loss', 'log2_loss', 'accuracy', 'precision', 'recall', 'f1', 'auc', 'pr_auc']
heldout_comparison_seed_rows = []
expected_row_index = np.sort(development_samples_df['row_index'].to_numpy(dtype=int))

for seed in SEEDS:
    transformer_predictions_df = (
        nuisance_seed_predictions_df
        .loc[nuisance_seed_predictions_df['seed'].eq(seed)]
        .sort_values('row_index')
        .reset_index(drop=True)
    )
    age_sex_predictions_df = age_sex_seed_prediction_map[seed].sort_values('row_index').reset_index(drop=True)
    rf_predictions_df = rf_seed_prediction_map[seed].sort_values('row_index').reset_index(drop=True)
    xgb_predictions_df = xgb_seed_prediction_map[seed].sort_values('row_index').reset_index(drop=True)
    scratch_predictions_df = scratch_transformer_seed_prediction_map[seed].sort_values('row_index').reset_index(drop=True)

    if not np.array_equal(transformer_predictions_df['row_index'].to_numpy(dtype=int), expected_row_index):
        raise ValueError(f'Transformer cross-fit predictions for seed {seed} do not match the development row_index set.')
    if not np.array_equal(age_sex_predictions_df['row_index'].to_numpy(dtype=int), expected_row_index):
        raise ValueError(f'Age/sex cross-fit predictions for seed {seed} do not match the development row_index set.')
    if not np.array_equal(rf_predictions_df['row_index'].to_numpy(dtype=int), expected_row_index):
        raise ValueError(f'RF cross-fit predictions for seed {seed} do not match the development row_index set.')
    if not np.array_equal(xgb_predictions_df['row_index'].to_numpy(dtype=int), expected_row_index):
        raise ValueError(f'XGBoost cross-fit predictions for seed {seed} do not match the development row_index set.')
    if not np.array_equal(scratch_predictions_df['row_index'].to_numpy(dtype=int), expected_row_index):
        raise ValueError(f'Scratch transformer cross-fit predictions for seed {seed} do not match the development row_index set.')

    mean_propensity_probs, mean_outcome_probs = build_mean_predictor_oof(seed)

    propensity_labels = transformer_predictions_df['treatment'].astype(int).to_numpy()
    outcome_labels = transformer_predictions_df['outcome'].astype(int).to_numpy()

    propensity_metrics = build_comparison_df([
        ('Mean predictor', classification_metrics_from_probabilities(
            propensity_labels,
            mean_propensity_probs,
            pos_weight=None,
        )),
        ('Age + sex', classification_metrics_from_probabilities(
            propensity_labels,
            age_sex_predictions_df['propensity'].to_numpy(dtype=float),
            pos_weight=None,
        )),
        ('Random forest', classification_metrics_from_probabilities(
            propensity_labels,
            rf_predictions_df['propensity'].to_numpy(dtype=float),
            pos_weight=None,
        )),
        ('XGBoost', classification_metrics_from_probabilities(
            propensity_labels,
            xgb_predictions_df['propensity'].to_numpy(dtype=float),
            pos_weight=None,
        )),
        ('NP-Transformer', classification_metrics_from_probabilities(
            propensity_labels,
            scratch_predictions_df['propensity'].to_numpy(dtype=float),
            pos_weight=None,
        )),
        ('ITE-Transformer', classification_metrics_from_probabilities(
            propensity_labels,
            transformer_predictions_df['propensity'].to_numpy(dtype=float),
            pos_weight=None,
        )),
    ])
    propensity_metrics['seed'] = seed
    propensity_metrics['task'] = 'propensity'
    heldout_comparison_seed_rows.extend(propensity_metrics.to_dict('records'))

    outcome_metrics = build_comparison_df([
        ('Mean predictor', classification_metrics_from_probabilities(
            outcome_labels,
            mean_outcome_probs,
            pos_weight=None,
        )),
        ('Age + sex', classification_metrics_from_probabilities(
            outcome_labels,
            observed_outcome_probabilities(age_sex_predictions_df),
            pos_weight=None,
        )),
        ('Random forest', classification_metrics_from_probabilities(
            outcome_labels,
            observed_outcome_probabilities(rf_predictions_df),
            pos_weight=None,
        )),
        ('XGBoost', classification_metrics_from_probabilities(
            outcome_labels,
            observed_outcome_probabilities(xgb_predictions_df),
            pos_weight=None,
        )),
        ('NP-Transformer', classification_metrics_from_probabilities(
            outcome_labels,
            observed_outcome_probabilities(scratch_predictions_df),
            pos_weight=None,
        )),
        ('ITE-Transformer', classification_metrics_from_probabilities(
            outcome_labels,
            observed_outcome_probabilities(transformer_predictions_df),
            pos_weight=None,
        )),
    ])
    outcome_metrics['seed'] = seed
    outcome_metrics['task'] = 'outcome'
    heldout_comparison_seed_rows.extend(outcome_metrics.to_dict('records'))

heldout_comparison_seed_metrics_df = pd.DataFrame(heldout_comparison_seed_rows)
heldout_comparison_metric_summary_df = summarize_seed_distribution(
    heldout_comparison_seed_metrics_df,
    group_cols=['task', 'model'],
    value_cols=comparison_metric_cols,
)

propensity_heldout_comparison_df = sort_models(
    heldout_comparison_metric_summary_df.loc[
        heldout_comparison_metric_summary_df['task'].eq('propensity')
    ].reset_index(drop=True)
)
outcome_heldout_comparison_df = sort_models(
    heldout_comparison_metric_summary_df.loc[
        heldout_comparison_metric_summary_df['task'].eq('outcome')
    ].reset_index(drop=True)
)

print('Held-out cross-fit comparisons with 95% confidence intervals across seeds')
display(propensity_heldout_comparison_df)
display(outcome_heldout_comparison_df)

plot_comparison_metrics(propensity_heldout_comparison_df, 'propensity')
plot_comparison_metrics(outcome_heldout_comparison_df, 'outcome')




### First-stage probability calibration and overlap

Reliability curves compare average predicted probabilities with observed event frequencies within bins. The diagonal denotes agreement. Examine propensity and factual-outcome calibration separately; systematic deviations can affect the pseudo-outcomes even when discrimination is strong.

The propensity distributions compare treatment groups. Limited overlap or concentrations near zero and one indicate sensitivity to large inverse-probability factors. Read these plots together with the clipping threshold and pseudo-outcome distributions. All use out-of-fold development predictions.

Calibration and overlap are useful diagnostics, but cannot establish that all confounders have been measured.


In [ ]:
import seaborn as sns


def aggregate_seed_prediction_map(prediction_map, value_columns):
    seed_frames = []
    for seed, seed_df in prediction_map.items():
        frame = seed_df[['row_index', 'person_id', 'treatment', 'outcome', *value_columns]].copy()
        frame['seed'] = seed
        seed_frames.append(frame)
    combined_df = pd.concat(seed_frames, ignore_index=True)
    summary_df = summarize_seed_distribution(
        combined_df,
        group_cols=['row_index', 'person_id', 'treatment', 'outcome'],
        value_cols=value_columns,
    )
    return combined_df, summary_df


transformer_seed_propensity_long_df, transformer_propensity_summary_df = aggregate_seed_prediction_map(
    {
        seed: nuisance_seed_predictions_df.loc[nuisance_seed_predictions_df['seed'].eq(seed)].copy()
        for seed in SEEDS
    },
    ['propensity'],
)
age_sex_seed_propensity_long_df, age_sex_propensity_summary_df = aggregate_seed_prediction_map(
    age_sex_seed_prediction_map,
    ['propensity'],
)
rf_seed_propensity_long_df, rf_propensity_summary_df = aggregate_seed_prediction_map(
    rf_seed_prediction_map,
    ['propensity'],
)
xgb_seed_propensity_long_df, xgb_propensity_summary_df = aggregate_seed_prediction_map(
    xgb_seed_prediction_map,
    ['propensity'],
)
scratch_transformer_seed_propensity_long_df, scratch_transformer_propensity_summary_df = aggregate_seed_prediction_map(
    scratch_transformer_seed_prediction_map,
    ['propensity'],
)

propensity_diagnostic_df = transformer_propensity_summary_df.rename(columns={
    'propensity_mean': 'propensity',
    'propensity_std': 'propensity_seed_std',
}).copy()
propensity_diagnostic_df['treatment'] = propensity_diagnostic_df['treatment'].astype(int)
propensity_diagnostic_df['propensity_clipped'] = propensity_diagnostic_df['propensity'].clip(1e-6, 1 - 1e-6)
propensity_diagnostic_df = propensity_diagnostic_df.merge(
    age_sex_propensity_summary_df[['row_index', 'propensity_mean', 'propensity_std']].rename(columns={
        'propensity_mean': 'age_sex_propensity',
        'propensity_std': 'age_sex_propensity_seed_std',
    }),
    on='row_index',
    how='left',
    validate='one_to_one',
)
propensity_diagnostic_df = propensity_diagnostic_df.merge(
    rf_propensity_summary_df[['row_index', 'propensity_mean', 'propensity_std']].rename(columns={
        'propensity_mean': 'rf_propensity',
        'propensity_std': 'rf_propensity_seed_std',
    }),
    on='row_index',
    how='left',
    validate='one_to_one',
)
propensity_diagnostic_df = propensity_diagnostic_df.merge(
    xgb_propensity_summary_df[['row_index', 'propensity_mean', 'propensity_std']].rename(columns={
        'propensity_mean': 'xgb_propensity',
        'propensity_std': 'xgb_propensity_seed_std',
    }),
    on='row_index',
    how='left',
    validate='one_to_one',
)
propensity_diagnostic_df = propensity_diagnostic_df.merge(
    scratch_transformer_propensity_summary_df[['row_index', 'propensity_mean', 'propensity_std']].rename(columns={
        'propensity_mean': 'scratch_transformer_propensity',
        'propensity_std': 'scratch_transformer_propensity_seed_std',
    }),
    on='row_index',
    how='left',
    validate='one_to_one',
)
propensity_diagnostic_df['age_sex_propensity_clipped'] = propensity_diagnostic_df['age_sex_propensity'].clip(1e-6, 1 - 1e-6)
propensity_diagnostic_df['rf_propensity_clipped'] = propensity_diagnostic_df['rf_propensity'].clip(1e-6, 1 - 1e-6)
propensity_diagnostic_df['xgb_propensity_clipped'] = propensity_diagnostic_df['xgb_propensity'].clip(1e-6, 1 - 1e-6)
propensity_diagnostic_df['scratch_transformer_propensity_clipped'] = propensity_diagnostic_df['scratch_transformer_propensity'].clip(1e-6, 1 - 1e-6)

treated_mask = propensity_diagnostic_df['treatment'].eq(1)
control_mask = propensity_diagnostic_df['treatment'].eq(0)

overlap_plot_specs = [
    ('propensity_clipped', 'Propensity score overlap'),
    ('age_sex_propensity_clipped', 'Age + sex propensity score overlap'),
    ('rf_propensity_clipped', 'Random forest propensity score overlap'),
    ('xgb_propensity_clipped', 'XGBoost propensity score overlap'),
    ('scratch_transformer_propensity_clipped', 'NP-Transformer propensity score overlap'),
]

for propensity_column, plot_title in overlap_plot_specs:
    fig, ax = plt.subplots(figsize=(6, 4))
    sns.kdeplot(
        data=propensity_diagnostic_df.loc[treated_mask, propensity_column],
        label=TREATMENT_NAME,
        fill=True,
        color='#5B8DEF',
        common_norm=False,
        alpha=0.35,
        ax=ax,
    )
    sns.kdeplot(
        data=propensity_diagnostic_df.loc[control_mask, propensity_column],
        label=COMPARATOR_NAME,
        fill=True,
        color='#D06A4C',
        common_norm=False,
        alpha=0.35,
        ax=ax,
    )
    ax.set_title(plot_title)
    ax.set_xlabel('predicted treatment probability')
    ax.set_ylabel('density')
    ax.tick_params(axis='x', which='both', length=0)
    ax.tick_params(axis='y', which='both', length=0)
    ax.spines[['right', 'top']].set_visible(False)
    ax.spines['left'].set_color('grey')
    ax.spines['bottom'].set_color('grey')
    ax.legend(frameon=False)
    output_path = METRIC_PLOT_DIR / f'overlap_{safe_filename_token(propensity_column)}.svg'
    fig.savefig(output_path, format='svg', bbox_inches='tight')
    print(f'Saved {output_path}')
    plt.show()

calibration_seed_frames = []
calibration_prediction_specs = [
    ('Age + sex', age_sex_seed_propensity_long_df),
    ('Random forest', rf_seed_propensity_long_df),
    ('XGBoost', xgb_seed_propensity_long_df),
    ('NP-Transformer', scratch_transformer_seed_propensity_long_df),
    ('ITE-Transformer', transformer_seed_propensity_long_df),
]
calibration_bin_edges = np.linspace(0.0, 1.0, 11)
CALIBRATION_AXIS_LIMITS = (-0.03, 1.03)
CALIBRATION_BAND_ALPHA = 0.08

for model_name, long_df in calibration_prediction_specs:
    seed_df = long_df.copy()
    seed_df['probability'] = seed_df['propensity'].clip(1e-6, 1 - 1e-6)
    seed_df['calibration_bin'] = pd.cut(
        seed_df['probability'],
        bins=calibration_bin_edges,
        labels=False,
        include_lowest=True,
        duplicates='drop',
    )
    seed_bin_summary = (
        seed_df.dropna(subset=['calibration_bin'])
        .assign(calibration_bin=lambda frame: frame['calibration_bin'].astype(int))
        .groupby(['seed', 'calibration_bin'], observed=True)
        .agg(
            mean_predicted_propensity=('probability', 'mean'),
            observed_treatment_rate=('treatment', 'mean'),
            n=('treatment', 'size'),
        )
        .reset_index()
    )
    seed_bin_summary['model'] = model_name
    calibration_seed_frames.append(seed_bin_summary)

propensity_calibration_by_seed_df = pd.concat(calibration_seed_frames, ignore_index=True)
propensity_calibration_summary_df = summarize_seed_distribution(
    propensity_calibration_by_seed_df,
    group_cols=['model', 'calibration_bin'],
    value_cols=['mean_predicted_propensity', 'observed_treatment_rate', 'n'],
)

for model_name in ['Age + sex', 'Random forest', 'XGBoost', 'NP-Transformer', 'ITE-Transformer']:
    model_df = propensity_calibration_summary_df.loc[
        propensity_calibration_summary_df['model'].eq(model_name)
    ].sort_values('calibration_bin')
    if model_df.empty:
        continue

    fig, ax = plt.subplots(figsize=(4, 4), constrained_layout=True)
    ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1, color='gray', label='Ideal calibration')
    ax.spines[['right', 'top']].set_visible(False)
    ax.spines['left'].set_color('grey')
    ax.spines['bottom'].set_color('grey')

    ax.tick_params(axis='x', which='both', length=0)
    ax.tick_params(axis='y', which='both', length=0)
    ax.set_xlim(*CALIBRATION_AXIS_LIMITS)
    ax.set_ylim(*CALIBRATION_AXIS_LIMITS)
    ax.set_aspect('equal', adjustable='box')


    x = model_df['mean_predicted_propensity_mean'].to_numpy(dtype=float)
    y = model_df['observed_treatment_rate_mean'].to_numpy(dtype=float)
    y_low = model_df['observed_treatment_rate_ci_low'].to_numpy(dtype=float)
    y_high = model_df['observed_treatment_rate_ci_high'].to_numpy(dtype=float)
    ax.plot(
        x,
        y,
        marker='o',
        markersize=6,
        markeredgecolor='white', markeredgewidth=1,
        linewidth=2,
        label=model_name,
        color=MODEL_COLORS.get(model_name),
    )
    ax.fill_between(x, y_low, y_high, color=MODEL_COLORS.get(model_name), alpha=CALIBRATION_BAND_ALPHA)
    #ax.set_title(f'Propensity model calibration - {model_name}')
    ax.set_xlabel('Mean predicted propensity')
    ax.set_ylabel('Observed treatment frequency')
    ax.legend(edgecolor='white')
    output_path = METRIC_PLOT_DIR / f'propensity_calibration_{safe_filename_token(model_name)}.svg'
    fig.savefig(output_path, format='svg', bbox_inches='tight', pad_inches=0.12)
    print(f'Saved {output_path}')
    plt.show()

required_outcome_calibration_columns = [
    'row_index',
    'person_id',
    'treatment',
    'outcome',
    'outcome_probability_treatment_0',
    'outcome_probability_treatment_1',
]
OUTCOME_CALIBRATION_BIN_EDGES = np.linspace(0.0, 1.0, 8)


def build_outcome_calibration_long_df(prediction_map):
    seed_frames = []
    for seed, seed_df in prediction_map.items():
        frame = seed_df[required_outcome_calibration_columns].copy()
        frame['seed'] = seed
        frame['treatment'] = frame['treatment'].astype(int)
        frame['outcome'] = frame['outcome'].astype(float)
        frame['outcome_probability_treatment_0'] = frame['outcome_probability_treatment_0'].astype(float)
        frame['outcome_probability_treatment_1'] = frame['outcome_probability_treatment_1'].astype(float)
        frame['predicted_factual_outcome_probability'] = np.where(
            frame['treatment'].eq(1),
            frame['outcome_probability_treatment_1'],
            frame['outcome_probability_treatment_0'],
        )
        seed_frames.append(frame)
    return pd.concat(seed_frames, ignore_index=True)

outcome_calibration_prediction_specs = [
    ('Age + sex', age_sex_seed_prediction_map),
    ('Random forest', rf_seed_prediction_map),
    ('XGBoost', xgb_seed_prediction_map),
    ('NP-Transformer', scratch_transformer_seed_prediction_map),
    ('ITE-Transformer', {
        seed: nuisance_seed_predictions_df.loc[nuisance_seed_predictions_df['seed'].eq(seed)].copy()
        for seed in SEEDS
    }),
]

outcome_calibration_overall_rows = []
outcome_calibration_seed_frames = []

for model_name, prediction_map in outcome_calibration_prediction_specs:
    outcome_long_df = build_outcome_calibration_long_df(prediction_map)

    for seed in SEEDS:
        seed_df = (
            outcome_long_df.loc[
                outcome_long_df['seed'].eq(seed)
            ]
            .copy()
            .reset_index(drop=True)
        )
        if seed_df.empty:
            raise ValueError(
                f'Outcome calibration requires at least one held-out row for seed {seed} and model {model_name}.'
            )

        seed_df['probability'] = seed_df['predicted_factual_outcome_probability'].clip(1e-6, 1 - 1e-6)
        outcome_calibration_overall_rows.append({
            'model': model_name,
            'seed': seed,
            'split': 'heldout',
            'n': len(seed_df),
            'mean_predicted_outcome_probability': float(seed_df['probability'].mean()),
            'mean_observed_outcome': float(seed_df['outcome'].mean()),
            'calibration_error': float(seed_df['outcome'].mean() - seed_df['probability'].mean()),
        })

        seed_df['calibration_bin'] = pd.cut(
            seed_df['probability'],
            bins=OUTCOME_CALIBRATION_BIN_EDGES,
            labels=False,
            include_lowest=True,
            duplicates='drop',
        )
        seed_bin_summary = (
            seed_df.dropna(subset=['calibration_bin'])
            .assign(calibration_bin=lambda frame: frame['calibration_bin'].astype(int))
            .groupby(['seed', 'calibration_bin'], observed=True)
            .agg(
                mean_predicted_outcome_probability=('probability', 'mean'),
                observed_outcome_rate=('outcome', 'mean'),
                n=('outcome', 'size'),
            )
            .reset_index()
        )
        seed_bin_summary['model'] = model_name
        outcome_calibration_seed_frames.append(seed_bin_summary)

outcome_calibration_overall_by_seed_df = pd.DataFrame(outcome_calibration_overall_rows)
outcome_calibration_overall_summary_df = summarize_seed_distribution(
    outcome_calibration_overall_by_seed_df,
    group_cols=['model', 'split'],
    value_cols=[
        'n',
        'mean_predicted_outcome_probability',
        'mean_observed_outcome',
        'calibration_error',
    ],
)
outcome_calibration_by_seed_df = pd.concat(outcome_calibration_seed_frames, ignore_index=True)
outcome_calibration_summary_df = summarize_seed_distribution(
    outcome_calibration_by_seed_df,
    group_cols=['model', 'calibration_bin'],
    value_cols=['mean_predicted_outcome_probability', 'observed_outcome_rate', 'n'],
)

for model_name in ['Age + sex', 'Random forest', 'XGBoost', 'NP-Transformer', 'ITE-Transformer']:
    model_df = outcome_calibration_summary_df.loc[
        outcome_calibration_summary_df['model'].eq(model_name)
    ].sort_values('calibration_bin')
    if model_df.empty:
        continue

    fig, ax = plt.subplots(figsize=(4, 4), constrained_layout=True)
    ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1, color='gray', label='Ideal calibration')
    ax.spines[['right', 'top']].set_visible(False)
    ax.spines['left'].set_color('grey')
    ax.spines['bottom'].set_color('grey')

    ax.tick_params(axis='x', which='both', length=0)
    ax.tick_params(axis='y', which='both', length=0)
    ax.set_xlim(*CALIBRATION_AXIS_LIMITS)
    ax.set_ylim(*CALIBRATION_AXIS_LIMITS)
    ax.set_aspect('equal', adjustable='box')


    x = model_df['mean_predicted_outcome_probability_mean'].to_numpy(dtype=float)
    y = model_df['observed_outcome_rate_mean'].to_numpy(dtype=float)
    y_low = model_df['observed_outcome_rate_ci_low'].to_numpy(dtype=float)
    y_high = model_df['observed_outcome_rate_ci_high'].to_numpy(dtype=float)
    ax.plot(
        x,
        y,
        marker='o',
        markersize=6,
        markeredgecolor='white', markeredgewidth=1,
        linewidth=2,
        label=model_name,
        color=MODEL_COLORS.get(model_name),
    )
    ax.fill_between(x, y_low, y_high, color=MODEL_COLORS.get(model_name), alpha=CALIBRATION_BAND_ALPHA)
    #ax.set_title(f'Outcome model calibration - {model_name}')
    ax.set_xlabel('Mean predicted outcome probability')
    ax.set_ylabel('Observed outcome frequency')
    ax.legend(edgecolor='white')
    output_path = METRIC_PLOT_DIR / f'outcome_calibration_{safe_filename_token(model_name)}.svg'
    fig.savefig(output_path, format='svg', bbox_inches='tight', pad_inches=0.12)
    print(f'Saved {output_path}')
    plt.show()

display(propensity_calibration_summary_df)
display(outcome_calibration_overall_by_seed_df)
display(outcome_calibration_overall_summary_df)
display(outcome_calibration_summary_df)


## 5. Load and summarise saved CATE predictions

The second stage learns a function of patient features from the first-stage pseudo-outcomes. For each seed, separate transformer CATE learners use IPW and AIPW targets. Saved predictions are loaded directly from the experiment runner. A development validation split selects the model.

The conditional-mean regression formulation is

$$\hat\tau\in\arg\min_f\sum_{i\in\mathcal D_{\mathrm{fit}}}\{\phi_i-f(X_i)\}^2.$$

The implemented transformer uses **Huber loss**, averaging equally across samples, to reduce sensitivity to extreme targets.

Prediction tables cover the full development population, including fitting rows. 

In [ ]:
def cate_pseudo_outcome_label(pseudo_outcome_column: str) -> str:
    return pseudo_outcome_column.removesuffix('_pseudo_outcome')


def cate_seed_dir(seed: int, pseudo_outcome_column: str = PSEUDO_OUTCOME_COLUMN) -> Path:
    return CATE_ROOT / cate_pseudo_outcome_label(pseudo_outcome_column) / f'seed_{seed}'


In [ ]:
cate_seed_predictions_by_pseudo = {}
cate_seed_metrics_by_pseudo = {}
cate_seed_metric_frames = []

propensity_df = propensity_df.drop(
    columns=[
        'cate_predicted_treatment_effect',
        'cate_predicted_treatment_effect_seed_std',
        'cate_minus_nuisance_ite',
        *[
            f'cate_predicted_treatment_effect_{cate_pseudo_outcome_label(column)}'
            for column in PSEUDO_OUTCOME_COLUMNS
        ],
        *[
            f'cate_predicted_treatment_effect_{cate_pseudo_outcome_label(column)}_seed_std'
            for column in PSEUDO_OUTCOME_COLUMNS
        ],
        *[
            f'cate_minus_nuisance_ite_{cate_pseudo_outcome_label(column)}'
            for column in PSEUDO_OUTCOME_COLUMNS
        ],
    ],
    errors='ignore',
)

for pseudo_outcome_column in PSEUDO_OUTCOME_COLUMNS:
    pseudo_outcome_label = cate_pseudo_outcome_label(pseudo_outcome_column)
    cate_seed_predictions = []
    cate_seed_metric_rows = []

    for seed in SEEDS:
        seed_dir = cate_seed_dir(seed, pseudo_outcome_column)
        predictions_df = pd.read_parquet(seed_dir / 'patient_cate_predictions.parquet')
        metrics_payload = json.loads((seed_dir / 'metrics.json').read_text())

        predictions_df['seed'] = seed
        predictions_df['pseudo_outcome_column'] = pseudo_outcome_column
        predictions_df['pseudo_outcome_label'] = pseudo_outcome_label
        cate_seed_predictions.append(predictions_df)

        development_split_name = 'development_population' if 'development_population' in metrics_payload else 'test'
        for split_name, split_metrics in [('best_validation', metrics_payload['best_validation']), (development_split_name, metrics_payload[development_split_name])]:
            cate_seed_metric_rows.append({
                'seed': seed,
                'split': split_name,
                'pseudo_outcome_column': pseudo_outcome_column,
                'pseudo_outcome_label': pseudo_outcome_label,
                **split_metrics,
            })

    cate_seed_predictions_df_for_pseudo = pd.concat(cate_seed_predictions, ignore_index=True)
    cate_seed_metrics_df_for_pseudo = pd.DataFrame(cate_seed_metric_rows).sort_values(['seed', 'split']).reset_index(drop=True)
    cate_seed_predictions_by_pseudo[pseudo_outcome_column] = cate_seed_predictions_df_for_pseudo
    cate_seed_metrics_by_pseudo[pseudo_outcome_column] = cate_seed_metrics_df_for_pseudo
    cate_seed_metric_frames.append(cate_seed_metrics_df_for_pseudo)

    prediction_column = f'cate_predicted_treatment_effect_{pseudo_outcome_label}'
    prediction_std_column = f'{prediction_column}_seed_std'
    cate_prediction_mean_df = (
        cate_seed_predictions_df_for_pseudo
        .groupby('row_index', as_index=False)['predicted_treatment_effect']
        .mean()
        .rename(columns={'predicted_treatment_effect': prediction_column})
    )
    cate_prediction_std_df = (
        cate_seed_predictions_df_for_pseudo
        .groupby('row_index', as_index=False)['predicted_treatment_effect']
        .std(ddof=1)
        .rename(columns={'predicted_treatment_effect': prediction_std_column})
    )

    propensity_df = (
        propensity_df
        .merge(cate_prediction_mean_df, on='row_index', how='left', validate='one_to_one')
        .merge(cate_prediction_std_df, on='row_index', how='left', validate='one_to_one')
    )
    propensity_df[f'cate_minus_nuisance_ite_{pseudo_outcome_label}'] = (
        propensity_df[prediction_column] - propensity_df['nuisance_individual_treatment_effect']
    )

cate_seed_predictions_df = cate_seed_predictions_by_pseudo[PSEUDO_OUTCOME_COLUMN]
cate_seed_metrics_df = cate_seed_metrics_by_pseudo[PSEUDO_OUTCOME_COLUMN]
cate_seed_metrics_all_df = pd.concat(cate_seed_metric_frames, ignore_index=True).sort_values(
    ['pseudo_outcome_label', 'seed', 'split']
).reset_index(drop=True)

default_cate_label = cate_pseudo_outcome_label(PSEUDO_OUTCOME_COLUMN)
propensity_df['cate_predicted_treatment_effect'] = propensity_df[
    f'cate_predicted_treatment_effect_{default_cate_label}'
]
propensity_df['cate_predicted_treatment_effect_seed_std'] = propensity_df[
    f'cate_predicted_treatment_effect_{default_cate_label}_seed_std'
]
propensity_df['individual_treatment_effect'] = propensity_df['cate_predicted_treatment_effect']
propensity_df['cate_minus_nuisance_ite'] = propensity_df[
    f'cate_minus_nuisance_ite_{default_cate_label}'
]

display(cate_seed_metrics_all_df)
display(propensity_df[[
    'row_index',
    'person_id',
    'nuisance_individual_treatment_effect',
    'cate_predicted_treatment_effect_ipw',
    'cate_predicted_treatment_effect_aipw',
    'cate_predicted_treatment_effect',
    'cate_predicted_treatment_effect_seed_std',
    'ipw_pseudo_outcome',
    'aipw_pseudo_outcome',
]].head(10))


### Experiment summaries

The following cell summarises first-stage nuisance estimates and second-stage transformer CATE metrics across seeds, and saves the aggregated predictions and metrics to `TABLE_DIR`. IPW and AIPW targets are reported separately.


In [ ]:
nuisance_metric_summary_df = (
    nuisance_seed_summary_df[['mean_propensity', 'mean_individual_treatment_effect', 'ipw_ate', 'aipw_ate', 'mean_abs_aipw_pseudo_outcome']]
    .apply(summarize_metric_series)
    .T
)
cate_metric_summary_by_pseudo = {}

population_summary_rows = [
    {
        'estimate': 'Nuisance mean ITE',
        'value': float(propensity_df['nuisance_individual_treatment_effect'].mean()),
    },
    {
        'estimate': 'IPW mean pseudo-outcome',
        'value': float(propensity_df['ipw_pseudo_outcome'].mean()),
    },
    {
        'estimate': 'AIPW mean pseudo-outcome',
        'value': float(propensity_df['aipw_pseudo_outcome'].mean()),
    },
]

aggregated_nuisance_path = TABLE_DIR / 'aggregated_nuisance_predictions.parquet'
nuisance_seed_metrics_path = TABLE_DIR / 'aggregated_nuisance_seed_summary.parquet'
aggregated_cate_paths = {}
cate_seed_metric_paths = {}

propensity_df.to_parquet(aggregated_nuisance_path, index=False)
nuisance_seed_summary_df.to_parquet(nuisance_seed_metrics_path, index=False)

for pseudo_outcome_column in PSEUDO_OUTCOME_COLUMNS:
    pseudo_outcome_label = cate_pseudo_outcome_label(pseudo_outcome_column)
    cate_metric_summary_by_pseudo[pseudo_outcome_column] = (
        cate_seed_metrics_by_pseudo[pseudo_outcome_column]
        .groupby('split')[['loss', 'rmse', 'mae', 'corr', 'prediction_mean', 'target_mean']]
        .apply(lambda frame: pd.concat({column: summarize_metric_series(frame[column]) for column in frame.columns}, axis=0))
        .unstack(level=-1)
    )
    population_summary_rows.append({
        'estimate': f'CATE mean predicted treatment effect ({pseudo_outcome_label.upper()})',
        'value': float(propensity_df[f'cate_predicted_treatment_effect_{pseudo_outcome_label}'].mean()),
    })
    aggregated_cate_paths[pseudo_outcome_column] = TABLE_DIR / f'aggregated_cate_predictions_{pseudo_outcome_label}.parquet'
    cate_seed_metric_paths[pseudo_outcome_column] = TABLE_DIR / f'aggregated_cate_seed_metrics_{pseudo_outcome_label}.parquet'
    cate_seed_predictions_by_pseudo[pseudo_outcome_column].to_parquet(
        aggregated_cate_paths[pseudo_outcome_column],
        index=False,
    )
    cate_seed_metrics_by_pseudo[pseudo_outcome_column].to_parquet(
        cate_seed_metric_paths[pseudo_outcome_column],
        index=False,
    )

population_summary_df = pd.DataFrame(population_summary_rows)
aggregated_cate_path = aggregated_cate_paths[PSEUDO_OUTCOME_COLUMN]
cate_seed_metrics_path = cate_seed_metric_paths[PSEUDO_OUTCOME_COLUMN]

display(population_summary_df)
display(nuisance_metric_summary_df)
for pseudo_outcome_column in PSEUDO_OUTCOME_COLUMNS:
    pseudo_outcome_label = cate_pseudo_outcome_label(pseudo_outcome_column)
    print(f'CATE metric summary ({pseudo_outcome_label.upper()})')
    display(cate_metric_summary_by_pseudo[pseudo_outcome_column])
print(f'Saved aggregated nuisance predictions to: {aggregated_nuisance_path}')
for pseudo_outcome_column in PSEUDO_OUTCOME_COLUMNS:
    pseudo_outcome_label = cate_pseudo_outcome_label(pseudo_outcome_column)
    print(
        f'Saved aggregated CATE predictions ({pseudo_outcome_label.upper()}) to: '
        f"{aggregated_cate_paths[pseudo_outcome_column]}"
    )
    print(
        f'Saved aggregated CATE seed metrics ({pseudo_outcome_label.upper()}) to: '
        f"{cate_seed_metric_paths[pseudo_outcome_column]}"
    )


## 6. Internal calibration of the CATE scores

For each seed, the next cell ranks development patients by the AIPW learner's predicted CATE and divides them into quantile bins. Within each bin, it compares the average predicted effect with the average cross-fit AIPW pseudo-outcome.

This analysis supports the internal consistency of the estimated treatment-effect heterogeneity. However, because the CATE learner was evaluated on the same data used for second-stage fitting, these results should be interpreted as an in-sample calibration check rather than evidence of out-of-sample generalizability.


In [ ]:
METRIC_PLOT_DIR.mkdir(parents=True, exist_ok=True)

def assign_in_fold_effect_bins(seed_df: pd.DataFrame, num_bins: int) -> pd.DataFrame:
    result = seed_df.copy()
    ranked_scores = result['cate_predicted_treatment_effect'].rank(method='first')
    result['effect_bin_index'] = (
        pd.qcut(ranked_scores, q=num_bins, labels=False, duplicates='drop').astype(int) + 1
    )
    if result['effect_bin_index'].nunique() != num_bins:
        raise ValueError(f'Could not form {num_bins} non-empty effect-score bins.')
    result['bin_count'] = int(num_bins)
    result['effect_bin'] = result['effect_bin_index'].map(
        lambda value: f'Q{int(value)}'
    )
    return result


in_fold_effect_score_assignment_frames = []
in_fold_effect_score_bin_rows = []

for seed in SEEDS:
    cate_seed_df = (
        cate_seed_predictions_by_pseudo[IN_FOLD_CATE_PSEUDO_OUTCOME_COLUMN]
        .loc[lambda frame: frame['seed'].eq(seed), ['row_index', 'predicted_treatment_effect']]
        .rename(columns={'predicted_treatment_effect': 'cate_predicted_treatment_effect'})
    )
    nuisance_seed_df = (
        nuisance_seed_predictions_df
        .loc[
            lambda frame: frame['seed'].eq(seed),
            ['row_index', 'person_id', 'treatment', 'outcome', 'aipw_pseudo_outcome'],
        ]
    )
    seed_df = cate_seed_df.merge(
        nuisance_seed_df,
        on='row_index',
        how='inner',
        validate='one_to_one',
    )
    if len(seed_df) != len(cate_seed_df) or len(seed_df) != len(nuisance_seed_df):
        raise ValueError(
            f'AIPW CATE and cross-fit nuisance rows do not align for seed {seed}: '
            f'{len(cate_seed_df)} CATE rows, {len(nuisance_seed_df)} nuisance rows, '
            f'{len(seed_df)} matched rows.'
        )
    seed_df['seed'] = int(seed)

    for num_bins in IN_FOLD_EFFECT_BIN_COUNTS:
        assignments_df = assign_in_fold_effect_bins(seed_df, num_bins=num_bins)
        in_fold_effect_score_assignment_frames.append(assignments_df)

        for (effect_bin_index, effect_bin), bin_df in assignments_df.groupby(
            ['effect_bin_index', 'effect_bin'], sort=True, observed=True
        ):
            treated_outcomes = bin_df.loc[bin_df['treatment'].eq(1), 'outcome']
            comparator_outcomes = bin_df.loc[bin_df['treatment'].eq(0), 'outcome']
            observed_ate = (
                float(treated_outcomes.mean() - comparator_outcomes.mean())
                if len(treated_outcomes) and len(comparator_outcomes)
                else np.nan
            )
            in_fold_effect_score_bin_rows.append({
                'seed': int(seed),
                'bin_count': int(num_bins),
                'effect_bin_index': int(effect_bin_index),
                'effect_bin': effect_bin,
                'n': int(len(bin_df)),
                'predicted_ite_mean': float(bin_df['cate_predicted_treatment_effect'].mean()),
                'aipw_ate': float(bin_df['aipw_pseudo_outcome'].mean()),
                'observed_ate': observed_ate,
            })

in_fold_effect_score_assignments_by_seed_df = pd.concat(
    in_fold_effect_score_assignment_frames,
    ignore_index=True,
)
in_fold_effect_score_bins_by_seed_df = pd.DataFrame(in_fold_effect_score_bin_rows)
in_fold_effect_score_curve_summary_df = summarize_seed_distribution(
    in_fold_effect_score_bins_by_seed_df,
    group_cols=['bin_count', 'effect_bin_index', 'effect_bin'],
    value_cols=['n', 'predicted_ite_mean', 'aipw_ate', 'observed_ate'],
).sort_values(['bin_count', 'effect_bin_index']).reset_index(drop=True)

print('In-fold effect-score summary (AIPW CATE only)')
display(
    in_fold_effect_score_bins_by_seed_df.sort_values(
        ['seed', 'bin_count', 'effect_bin_index']
    )
)
display(in_fold_effect_score_curve_summary_df)


def plot_in_fold_effect_score_curves(
    curve_df: pd.DataFrame,
    comparison_prefix: str,
    comparison_label: str,
    comparison_color: str,
    figure_title: str,
) -> None:
    fig, axes = plt.subplots(
        1,
        len(IN_FOLD_EFFECT_BIN_COUNTS),
        figsize=(6 * len(IN_FOLD_EFFECT_BIN_COUNTS), 5),
        sharey=True,
    )
    axes = np.atleast_1d(axes)
    all_values = np.concatenate([
        curve_df['predicted_ite_mean_ci_low'].to_numpy(dtype=float),
        curve_df['predicted_ite_mean_ci_high'].to_numpy(dtype=float),
        curve_df[f'{comparison_prefix}_ci_low'].to_numpy(dtype=float),
        curve_df[f'{comparison_prefix}_ci_high'].to_numpy(dtype=float),
    ])
    finite_values = all_values[np.isfinite(all_values)]
    if finite_values.size == 0:
        raise ValueError(f'No finite values are available for the {comparison_prefix} plot.')
    value_min = float(finite_values.min())
    value_max = float(finite_values.max())
    padding = 0.10 * max(value_max - value_min, 1e-6)

    for ax, num_bins in zip(axes, IN_FOLD_EFFECT_BIN_COUNTS):
        plot_df = curve_df.loc[curve_df['bin_count'].eq(num_bins)].sort_values('effect_bin_index')
        x = plot_df['effect_bin_index'].to_numpy(dtype=float)
        predicted_mean = plot_df['predicted_ite_mean_mean'].to_numpy(dtype=float)
        comparison_mean = plot_df[f'{comparison_prefix}_mean'].to_numpy(dtype=float)

        ax.plot(x, predicted_mean, marker='o', color='#5B8DEF', linewidth=2,
                label='Average predicted treatment effect')
        ax.fill_between(
            x,
            plot_df['predicted_ite_mean_ci_low'].to_numpy(dtype=float),
            plot_df['predicted_ite_mean_ci_high'].to_numpy(dtype=float),
            color='#5B8DEF', alpha=0.15,
        )
        ax.plot(x, comparison_mean, marker='o', color=comparison_color, linewidth=2,
                label=comparison_label)
        ax.fill_between(
            x,
            plot_df[f'{comparison_prefix}_ci_low'].to_numpy(dtype=float),
            plot_df[f'{comparison_prefix}_ci_high'].to_numpy(dtype=float),
            color=comparison_color, alpha=0.15,
        )
        ax.axhline(0.0, color='grey', linestyle='--', linewidth=1)
        #ax.set_title(f'In-fold effect-score calibration (AIPW CATE): {num_bins} bins')
        ax.set_xlabel('Treatment effect-score quantile bins')
        ax.set_xticks(x)
        ax.tick_params(axis='x', which='both', length=0)
        ax.tick_params(axis='y', which='both', length=0)
        ax.set_xticklabels(plot_df['effect_bin'])
        ax.set_xlim(0.75, num_bins + 0.25)
        ax.set_ylim(value_min - padding, value_max + padding)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('grey')
        ax.spines['bottom'].set_color('grey')
        #ax.grid(alpha=0.25)
        ax.legend(loc='best', frameon=False)

    axes[0].set_ylabel('Treatment effect')
    fig.suptitle(figure_title, y=1.02)
    fig.tight_layout()
    fig.savefig(METRIC_PLOT_DIR / "internal_effect_score_calibration.svg", format="svg", bbox_inches="tight")
    plt.show()


plot_in_fold_effect_score_curves(
    in_fold_effect_score_curve_summary_df,
    comparison_prefix='aipw_ate',
    comparison_label='AIPW ATE',
    comparison_color='#D06A4C',
    figure_title='',
)


# Patient-level assignments remain local; no W&B upload is performed.
in_fold_effect_score_bins_by_seed_df.to_csv(TABLE_DIR / 'internal_effect_score_bins_by_seed.csv', index=False)
in_fold_effect_score_curve_summary_df.to_csv(TABLE_DIR / 'internal_effect_score_summary.csv', index=False)
print(f'Figures: {FIGURE_DIR}')
print(f'Summary tables: {TABLE_DIR}')
